In [ ]:
# SPDX-FileCopyrightText: 2026 Mario Gemoll
# SPDX-License-Identifier: 0BSD

import os
import subprocess

def is_correct_repo() -> bool:
    try:
        result = subprocess.run(
            ["git", "remote", "get-url", "origin"],
            capture_output=True,
            text=True,
            check=True,
        )
        remote_url = result.stdout.strip()
        return remote_url in [
            "https://github.com/mariogemoll/reinforcement-learning.git",
            "git@github.com:mariogemoll/reinforcement-learning.git",
        ]
    except (subprocess.CalledProcessError, FileNotFoundError):
        return False


if not is_correct_repo():
    !git clone https://github.com/mariogemoll/reinforcement-learning.git

if not os.getcwd().endswith("reinforcement-learning/py"):
    %cd reinforcement-learning/py

In [ ]:
%pip install -q gymnax

In [ ]:
import jax
import jax.numpy as jnp
import gymnax
from policygradient import (
    calculate_returns, init_mlp_params, log_prob_trajectories, generate_rollouts
)
import matplotlib.pyplot as plt

In [ ]:
num_iterations = 400
num_rollouts = 32
max_steps = 500
lr = 1e-2
seed = 1

In [ ]:
def reinforce_with_baseline_loss(params, rollouts):
    log_probs = log_prob_trajectories(params, rollouts)
    returns = calculate_returns(rollouts)
    advantages = (returns - returns.mean()) / (returns.std() + 1e-8)
    return -jnp.mean(log_probs * advantages)

In [ ]:
# Build multi-seed runs for the visualization cell below (parallel over seeds)
seeds = jnp.array(range(5), dtype=jnp.int32)  # increase for smoother stats
loss_fn_for_seeds = reinforce_with_baseline_loss  # or reinforce_loss
env, env_params = gymnax.make("CartPole-v1")

def train_all_one_seed(params, key, loss_fn):
    def body(carry, _):
        params, key = carry
        key, subkey = jax.random.split(key)
        rollouts = generate_rollouts(
            subkey, env, env_params, params, num_rollouts, max_steps
        )
        loss, grad = jax.value_and_grad(loss_fn)(params, rollouts)
        params = jax.tree.map(lambda p, g: p - lr * g, params, grad)
        mean_ret = jnp.mean(calculate_returns(rollouts))
        return (params, key), (loss, mean_ret)

    (params, key), (loss_hist, ret_hist) = jax.lax.scan(
        body, (params, key), xs=None, length=num_iterations
    )
    return params, key, loss_hist, ret_hist

def train_all_seeds(params0, keys0, loss_fn):
    return jax.vmap(train_all_one_seed, in_axes=(0, 0, None))(params0, keys0, loss_fn)

jit_train_all_seeds = jax.jit(train_all_seeds, static_argnames=("loss_fn",))

# Per-seed initial keys and params
key_inits = jax.random.split(jax.random.PRNGKey(12345), seeds.shape[0])
params0 = jax.vmap(lambda k: init_mlp_params(k, [4, 8, 2]))(key_inits)
keys0 = jax.vmap(jax.random.PRNGKey)(seeds)

params_final, keys_final, loss_hist, seed_final_returns = jit_train_all_seeds(
    params0, keys0, loss_fn_for_seeds
)
_ = seed_final_returns.block_until_ready()

# Keep the same structure expected by the plotting cell
seed_runs = [
    {"seed": int(seeds[i]), "mean_returns": seed_final_returns[i]}
    for i in range(seeds.shape[0])
]

print(
    f"Prepared seed_runs with {len(seed_runs)} runs and "
    f"{seed_runs[0]['mean_returns'].shape[0]} iterations each (parallelized)."
)

In [ ]:
# Visualization
all_returns = jnp.stack([run["mean_returns"] for run in seed_runs], axis=0)
mean_return_curve = jnp.mean(all_returns, axis=0)
std_return_curve = jnp.std(all_returns, axis=0)

all_losses = loss_hist
mean_loss_curve = jnp.mean(all_losses, axis=0)
std_loss_curve = jnp.std(all_losses, axis=0)

x = jnp.arange(all_returns.shape[1])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Return curves
for run in seed_runs:
    axes[0].plot(
        run["mean_returns"], alpha=0.6, linewidth=1.2, label=f"seed {run['seed']}"
    )
axes[0].plot(
    mean_return_curve, linewidth=2.5, label="mean across seeds", color="black"
)
axes[0].fill_between(
    x,
    mean_return_curve - std_return_curve,
    mean_return_curve + std_return_curve,
    alpha=0.2,
    label="±1 std",
    color="gray",
)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Mean rollout return")
axes[0].set_title("Returns")
axes[0].legend(ncol=2, fontsize=8)

# Loss curves
for i in range(all_losses.shape[0]):
    axes[1].plot(all_losses[i], alpha=0.6, linewidth=1.2, label=f"seed {int(seeds[i])}")
axes[1].plot(mean_loss_curve, linewidth=2.5, label="mean across seeds", color="black")
axes[1].fill_between(
    x,
    mean_loss_curve - std_loss_curve,
    mean_loss_curve + std_loss_curve,
    alpha=0.2,
    label="±1 std",
    color="gray",
)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Loss")
axes[1].set_title("Loss")
axes[1].legend(ncol=2, fontsize=8)

fig.suptitle("CartPole REINFORCE: multi-seed training stability", y=1.03)
plt.tight_layout()
plt.show()